In [1]:
# ======================================================================
# CLEAN BER DATA: one ML-ready table, one row per BER certificate
# ======================================================================
# Input:  the 19 processed BER tables (processed_dataset\housing_energy_dataset)
# Output: clean_dataset\ber\ber_clean.csv
#
# What this table is for (from the proposal, Student 1):
#   - K-Prototypes clustering + FAMD   -> building features
#   - XGBoost regression               -> target: BerRating (kWh/m2/yr)
#   - Random Forest classification     -> target: is_B3_or_better
#
# Rules for this notebook:
#   - fix the data (spaces, types, hidden missing values, impossible values)
#   - do NOT fill missing values, scale or one-hot encode here. That happens
#     inside the model Pipeline on training data only (avoids data leakage)
#   - leave out leakage columns (outputs of the BER calculation)
#
# Needs 00_county_lookup.ipynb to be run first (uses county_lookup.csv)
# ======================================================================

import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

base = r'C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project\dataSet'

ber_folder = base + r'\processed_dataset\housing_energy_dataset'
clean      = base + r'\clean_dataset'

In [2]:
# Loading only the tables that hold useful inputs or the targets.
# Left out on purpose:
#   group_heating, group_boilers, other_energy, solar_pv, wind_turbines, floors
#   -> mostly empty or already summed up in other columns
#   most of results -> CO2Rating, EPC, CPC, RER, primary energy columns,
#   HeatLossIndicator are calculated from the same energy calculation as
#   BerRating, so using them as inputs would be leakage

def load(name):
    return pd.read_csv(ber_folder + '\\Final_datset_' + name + '.csv', low_memory=False)

assessment    = load('assessment')
results       = load('results')[['ber_id', 'BerRating', 'EnergyRating']]
geometry      = load('geometry')
fabric        = load('fabric')
ventilation   = load('ventilation')
space_heating = load('space_heating')
water_heating = load('water_heating')
solar_water   = load('solar_water')[['ber_id', 'SolarHotWaterHeating']]
renewables    = load('nzeb_renewables')[['ber_id', 'SolarPVEnergyOutput']]
walls         = load('walls')

ber_county    = load('county')
dwelling_type = load('dwelling_type')
fuel          = load('fuel')

print('assessment:', assessment.shape)
print('walls:     ', walls.shape)

assessment: (569058, 12)
walls:      (952212, 9)


In [3]:
# Picking the columns that describe the building itself.
# These are the things an assessor records about the house: size, insulation,
# ventilation, heating system. None of them are results of the rating.

assessment = assessment[['ber_id', 'county_id', 'dwelling_type_id', 'Year_of_Construction',
                         'TypeofRating', 'PurposeOfRating', 'DateOfAssessment']]

geometry = geometry[['ber_id', 'GroundFloorArea(sq m)', 'NoStoreys', 'WallArea', 'RoofArea',
                     'FloorArea', 'WindowArea', 'DoorArea', 'LivingAreaPercent',
                     'RoomInRoofArea', 'PredominantRoofType']]

fabric = fabric[['ber_id', 'UValueWall', 'UValueRoof', 'UValueFloor', 'UValueWindow', 'UvalueDoor',
                 'ThermalBridgingFactor', 'ThermalMassCategory', 'StructureType',
                 'SuspendedWoodenFloor', 'PercentageDraughtStripped', 'NoOfSidesSheltered',
                 'PermeabilityTest']]

ventilation = ventilation[['ber_id', 'NoOfChimneys', 'NoOfOpenFlues', 'NoOfFansAndVents',
                           'NoOfFluelessGasFires', 'DraftLobby', 'VentilationMethod',
                           'HeatExchangerEff']]

space_heating = space_heating[['ber_id', 'main_fuel_id', 'HSMainSystemEfficiency',
                               'HSSupplHeatFraction', 'HeatSystemControlCat',
                               'HeatSystemResponseCat', 'UndergroundHeating',
                               'WarmAirHeatingSystem', 'CombiBoiler']]
space_heating = space_heating.rename(columns={'main_fuel_id': 'space_fuel_id'})

water_heating = water_heating[['ber_id', 'main_fuel_id', 'WHMainSystemEff', 'WaterStorageVolume',
                               'InsulationType', 'CylinderStat']]
water_heating = water_heating.rename(columns={'main_fuel_id': 'water_fuel_id'})

In [4]:
# A house can have up to 3 wall types. For one row per BER, keeping the
# main wall type = the one with the biggest area.

walls = walls.sort_values(['ber_id', 'wall_area'], ascending=[True, False])
main_wall = walls.drop_duplicates(subset='ber_id', keep='first')
main_wall = main_wall[['ber_id', 'wall_type']].rename(columns={'wall_type': 'MainWallType'})

print('BERs with a main wall type:', len(main_wall))
main_wall.head()

BERs with a main wall type: 569056


,ber_id,MainWallType
0,1,Unknown
1,2,Unknown
2,3,Unknown
3,4,Other
4,5,Stone


In [5]:
# Joining everything back into one table using ber_id (like a SQL join).
# how='left' keeps every BER from the assessment table.

ber = assessment.merge(results, on='ber_id', how='left')
ber = ber.merge(geometry, on='ber_id', how='left')
ber = ber.merge(fabric, on='ber_id', how='left')
ber = ber.merge(ventilation, on='ber_id', how='left')
ber = ber.merge(space_heating, on='ber_id', how='left')
ber = ber.merge(water_heating, on='ber_id', how='left')
ber = ber.merge(solar_water, on='ber_id', how='left')
ber = ber.merge(renewables, on='ber_id', how='left')
ber = ber.merge(main_wall, on='ber_id', how='left')

print(ber.shape)   # should still be 569,058 rows

(569058, 54)


In [6]:
# Putting the names back in place of the ids (dwelling type and fuels),
# so the table is readable and the models can encode them later.

type_map = dict(zip(dwelling_type['dwelling_type_id'], dwelling_type['DwellingTypeDescr']))
fuel_map = dict(zip(fuel['fuel_id'], fuel['FuelName']))

ber['DwellingType']         = ber['dwelling_type_id'].map(type_map)
ber['MainSpaceHeatingFuel'] = ber['space_fuel_id'].map(fuel_map)
ber['MainWaterHeatingFuel'] = ber['water_fuel_id'].map(fuel_map)

ber = ber.drop(columns=['dwelling_type_id', 'space_fuel_id', 'water_fuel_id'])

print(ber['DwellingType'].value_counts(dropna=False))
print(ber['MainSpaceHeatingFuel'].value_counts(dropna=False))

DwellingType
Detached house            190994
Semi-detached house       145799
Mid-terrace house          70297
Mid-floor apartment        52043
End of terrace house       41837
Top-floor apartment        30068
Ground-floor apartment     29670
Maisonette                  8115
Basement Dwelling            232
Apartment                      3
Name: count, dtype: int64
MainSpaceHeatingFuel
Electricity                       216470
Heating Oil                       168089
Mains Gas                         154863
NaN                                12336
Solid Multi-Fuel                    7245
Bulk LPG (propane or butane)        5438
Manufactured Smokeless Fuel         2429
Wood Pellets (bulk supply for       1052
Bottled LPG                          406
House Coal                           199
Wood Logs                            164
Wood Pellets (in bags for seco       161
Sod Peat                             144
Wood Chips                            34
Peat Briquettes                     

In [7]:
# County: BER uses names like 'Co. Cork', 'Cork City', 'Dublin 6'.
# Using the shared county_lookup.csv so BER ends up with the same 26 county
# names (and county_id) as planning, PPR and rent.

ber_county_map = dict(zip(ber_county['county_id'], ber_county['CountyName']))
ber['ber_location'] = ber['county_id'].map(ber_county_map)
ber = ber.drop(columns=['county_id'])

lookup = pd.read_csv(clean + r'\county_lookup.csv')
lookup = lookup[lookup['source'] == 'ber'][['raw_name', 'county', 'county_id']]

ber = ber.merge(lookup, left_on='ber_location', right_on='raw_name', how='left')
ber = ber.drop(columns=['raw_name'])

print('Rows without a county:', ber['county'].isnull().sum())   # should be 0
print(ber['county'].nunique(), 'counties')

Rows without a county: 0
26 counties


In [8]:
# Text columns: SEAI pads a lot of values with spaces at the end
# ('Existing       ', 'NO  ', 'Masonry   '). Removing them.
# Also some 'missing' values are hidden as text: '' or 'Please select'.
# Turning those into real missing values (NaN).

text_cols = []
for col in ber.columns:
    if ber[col].dtype == 'object' or str(ber[col].dtype) == 'str':
        text_cols.append(col)

for col in text_cols:
    ber[col] = ber[col].str.strip()
    ber[col] = ber[col].replace(['', 'Please select'], np.nan)

print('Text columns cleaned:', text_cols)

Text columns cleaned: ['TypeofRating', 'PurposeOfRating', 'DateOfAssessment', 'EnergyRating', 'PredominantRoofType', 'ThermalMassCategory', 'StructureType', 'SuspendedWoodenFloor', 'PermeabilityTest', 'DraftLobby', 'VentilationMethod', 'UndergroundHeating', 'WarmAirHeatingSystem', 'CombiBoiler', 'InsulationType', 'CylinderStat', 'SolarHotWaterHeating', 'MainWallType', 'DwellingType', 'MainSpaceHeatingFuel', 'MainWaterHeatingFuel', 'ber_location', 'county']


In [9]:
# YES / NO columns -> 1 / 0, so models can use them directly.
# Only changing columns where every value is YES, NO, Yes or No.

yes_no_cols = []
for col in text_cols:
    values = set(ber[col].dropna().unique())
    if len(values) > 0 and values.issubset({'YES', 'NO', 'Yes', 'No'}):
        ber[col] = ber[col].map({'YES': 1, 'Yes': 1, 'NO': 0, 'No': 0})
        yes_no_cols.append(col)

print('Changed to 1/0:', yes_no_cols)

Changed to 1/0: ['PermeabilityTest', 'DraftLobby', 'UndergroundHeating', 'WarmAirHeatingSystem', 'CylinderStat', 'SolarHotWaterHeating']


In [10]:
# Dates come back as text from CSV, so converting again.
# Adding year and quarter (e.g. 2021Q1) in the same format as PPR, rent and
# planning, for the county + quarter join later.

ber['DateOfAssessment'] = pd.to_datetime(ber['DateOfAssessment'])
ber['year'] = ber['DateOfAssessment'].dt.year
ber['quarter'] = ber['DateOfAssessment'].dt.to_period('Q').astype(str)

print(ber['year'].value_counts().sort_index())

year
2021     71878
2022    102107
2023    113180
2024    135072
2025    146821
Name: count, dtype: int64


In [11]:
# Year of construction goes up to 2073, which is impossible.
# A few 2026 values are fine (provisional ratings for homes not built yet),
# so only years after 2026 become missing.

print('Years after 2026:', (ber['Year_of_Construction'] > 2026).sum())
print(ber[ber['Year_of_Construction'] > 2026][['ber_id', 'Year_of_Construction', 'TypeofRating']])

ber.loc[ber['Year_of_Construction'] > 2026, 'Year_of_Construction'] = np.nan

Years after 2026: 2
       ber_id  Year_of_Construction TypeofRating
17370   17371                  2073     Existing
35732   35733                  2033     Existing


In [12]:
# Looking at the energy rating labels after removing spaces.
# Expected the usual 15 bands (A1 ... G), but there are also A, A0, B, C, D, E.

print(ber['EnergyRating'].value_counts().sort_index())

EnergyRating
A        355
A0       269
A1     17329
A2    129646
A3     36669
B        152
B1     28671
B2     40586
B3     58609
C         97
C1     51106
C2     47054
C3     38565
D         34
D1     32467
D2     26744
E         11
E1     15774
E2     12229
F      12952
G      19739
Name: count, dtype: int64


In [13]:
# The odd labels only show up from 2021 and grow a lot in 2024-2025,
# and their BerRating values sit in clear ranges (A0 lowest, then A, B, C...).
# So they look like a different (newer) rating scale, not typing mistakes.
# Keeping the rows, but marking which scale each label belongs to.

old_scale = ['A1', 'A2', 'A3', 'B1', 'B2', 'B3', 'C1', 'C2', 'C3',
             'D1', 'D2', 'E1', 'E2', 'F', 'G']

ber['rating_scale'] = 'other'
ber.loc[ber['EnergyRating'].isin(old_scale), 'rating_scale'] = 'old_15_band'

print(ber['rating_scale'].value_counts())
print()
print(ber.groupby(['EnergyRating'])['BerRating'].agg(['count', 'min', 'max']).sort_values('min'))

rating_scale
old_15_band    568140
other             918
Name: count, dtype: int64

               count     min      max
EnergyRating                         
A1             17329 -472.99    25.00
A0               269  -39.34    42.00
A                355  -30.58    74.99
A2            129646   25.00    50.00
A3             36669   50.00    75.00
B1             28671   75.00   100.00
B                152   75.29   149.80
B2             40586  100.00   125.00
B3             58609  125.00   150.00
C1             51106  150.00   175.00
C                 97  151.32   222.31
C2             47054  175.00   200.00
C3             38565  200.00   225.00
D1             32467  225.00   260.00
D                 34  227.27   270.02
D2             26744  260.00   300.00
E                 11  282.06   316.60
E1             15774  300.02   340.00
F              12952  326.68   450.00
E2             12229  340.01   379.99
G              19739  406.05  4000.65


In [14]:
# Classification target: is the home B3 or better?
# Homes on the 15-band scale: A1, A2, A3, B1, B2, B3 -> 1, everything else -> 0
# (using the letters here, because a few homes sit exactly on 150 and the
#  letter is the official answer for them)
# Homes on the other scale (A0, A, B, C, D, E) have no B3 letter, so for those
# the BerRating cut-off is used instead: B3 or better = up to 150 kWh/m2/yr.

b3_or_better = ['A1', 'A2', 'A3', 'B1', 'B2', 'B3']

ber['is_B3_or_better'] = 0
ber.loc[ber['EnergyRating'].isin(b3_or_better), 'is_B3_or_better'] = 1
ber.loc[(ber['rating_scale'] == 'other') & (ber['BerRating'] <= 150), 'is_B3_or_better'] = 1

# Checking the labels
print(pd.crosstab(ber['EnergyRating'], ber['is_B3_or_better']))
print()
print(ber['is_B3_or_better'].value_counts(normalize=True).round(3))

is_B3_or_better      0       1
EnergyRating                  
A                    0     355
A0                   0     269
A1                   0   17329
A2                   0  129646
A3                   0   36669
B                    0     152
B1                   0   28671
B2                   0   40586
B3                   0   58609
C                   97       0
C1               51106       0
C2               47054       0
C3               38565       0
D                   34       0
D1               32467       0
D2               26744       0
E                   11       0
E1               15774       0
E2               12229       0
F                12952       0
G                19739       0

is_B3_or_better
1    0.549
0    0.451
Name: proportion, dtype: float64


In [15]:
# BerRating range check. Negative values are homes that make more energy than
# they use (e.g. solar panels), so they are real and kept.
# Very high values are kept too; outliers are dealt with at the modelling stage.

print(ber['BerRating'].describe().round(2))
print('Negative BerRating:', (ber['BerRating'] < 0).sum())

count    569058.00
mean        160.10
std         140.19
min        -472.99
25%          48.50
50%         140.12
75%         210.87
max        4000.65
Name: BerRating, dtype: float64
Negative BerRating: 3666


In [16]:
# Tidying column names and order: keys first, then building features,
# then the targets at the end.

ber = ber.rename(columns={'GroundFloorArea(sq m)': 'TotalFloorArea'})

key_cols = ['ber_id', 'county', 'county_id', 'ber_location', 'DateOfAssessment', 'year', 'quarter']
target_cols = ['BerRating', 'EnergyRating', 'rating_scale', 'is_B3_or_better']

feature_cols = []
for col in ber.columns:
    if col not in key_cols and col not in target_cols:
        feature_cols.append(col)

ber = ber[key_cols + feature_cols + target_cols]

print(len(feature_cols), 'feature columns:')
print(feature_cols)

49 feature columns:
['Year_of_Construction', 'TypeofRating', 'PurposeOfRating', 'TotalFloorArea', 'NoStoreys', 'WallArea', 'RoofArea', 'FloorArea', 'WindowArea', 'DoorArea', 'LivingAreaPercent', 'RoomInRoofArea', 'PredominantRoofType', 'UValueWall', 'UValueRoof', 'UValueFloor', 'UValueWindow', 'UvalueDoor', 'ThermalBridgingFactor', 'ThermalMassCategory', 'StructureType', 'SuspendedWoodenFloor', 'PercentageDraughtStripped', 'NoOfSidesSheltered', 'PermeabilityTest', 'NoOfChimneys', 'NoOfOpenFlues', 'NoOfFansAndVents', 'NoOfFluelessGasFires', 'DraftLobby', 'VentilationMethod', 'HeatExchangerEff', 'HSMainSystemEfficiency', 'HSSupplHeatFraction', 'HeatSystemControlCat', 'HeatSystemResponseCat', 'UndergroundHeating', 'WarmAirHeatingSystem', 'CombiBoiler', 'WHMainSystemEff', 'WaterStorageVolume', 'InsulationType', 'CylinderStat', 'SolarHotWaterHeating', 'SolarPVEnergyOutput', 'MainWallType', 'DwellingType', 'MainSpaceHeatingFuel', 'MainWaterHeatingFuel']


In [17]:
# Missing values left in the clean table (% empty per column).
# Not filling them here; the model Pipeline will handle them on training data.

missing = (ber.isnull().mean() * 100).round(1)
missing[missing > 0].sort_values(ascending=False)

PredominantRoofType          11.9
StructureType                 8.2
SuspendedWoodenFloor          4.1
PercentageDraughtStripped     4.1
NoOfSidesSheltered            4.1
PermeabilityTest              4.1
NoOfChimneys                  4.1
NoOfOpenFlues                 4.1
NoOfFansAndVents              4.1
NoOfFluelessGasFires          4.1
DraftLobby                    4.1
VentilationMethod             4.1
HeatExchangerEff              4.1
HSMainSystemEfficiency        2.2
HSSupplHeatFraction           2.2
WHMainSystemEff               2.2
MainSpaceHeatingFuel          2.2
MainWaterHeatingFuel          2.2
dtype: float64

In [18]:
ber.info()
ber.head()

<class 'pandas.DataFrame'>
RangeIndex: 569058 entries, 0 to 569057
Data columns (total 60 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   ber_id                     569058 non-null  int64         
 1   county                     569058 non-null  str           
 2   county_id                  569058 non-null  int64         
 3   ber_location               569058 non-null  str           
 4   DateOfAssessment           569058 non-null  datetime64[us]
 5   year                       569058 non-null  int32         
 6   quarter                    569058 non-null  str           
 7   Year_of_Construction       569056 non-null  float64       
 8   TypeofRating               569058 non-null  str           
 9   PurposeOfRating            569058 non-null  str           
 10  TotalFloorArea             569058 non-null  float64       
 11  NoStoreys                  569058 non-null  int64         
 12 

,ber_id,county,county_id,ber_location,DateOfAssessment,year,quarter,Year_of_Construction,TypeofRating,PurposeOfRating,TotalFloorArea,NoStoreys,WallArea,RoofArea,FloorArea,WindowArea,DoorArea,LivingAreaPercent,RoomInRoofArea,PredominantRoofType,UValueWall,UValueRoof,UValueFloor,UValueWindow,UvalueDoor,ThermalBridgingFactor,ThermalMassCategory,StructureType,SuspendedWoodenFloor,PercentageDraughtStripped,NoOfSidesSheltered,PermeabilityTest,NoOfChimneys,NoOfOpenFlues,NoOfFansAndVents,NoOfFluelessGasFires,DraftLobby,VentilationMethod,HeatExchangerEff,HSMainSystemEfficiency,HSSupplHeatFraction,HeatSystemControlCat,HeatSystemResponseCat,UndergroundHeating,WarmAirHeatingSystem,CombiBoiler,WHMainSystemEff,WaterStorageVolume,InsulationType,CylinderStat,SolarHotWaterHeating,SolarPVEnergyOutput,MainWallType,DwellingType,MainSpaceHeatingFuel,MainWaterHeatingFuel,BerRating,EnergyRating,rating_scale,is_B3_or_better
0,1,Kildare,9,Co. Kildare,2021-01-01,2021,2021Q1,2000.0,Existing,Private Letting,25.65,1,50.75,25.65,25.65,3.47,1.85,45.61,0.0,Pitch Roof-Insul.on Rafter,0.55,0.36,0.52,2.80,3.00,0.15,Medium,Masonry,No,100.0,2.0,0.0,0.0,0.0,1.0,0.0,1.0,Natural vent.,0.0,85.60,0.0,1,1,0,0,None,85.60,110.0,Factory Insulated,0,0,0.00,Unknown,Detached house,Heating Oil,Heating Oil,339.72,E1,old_15_band,0
1,2,Kildare,9,Co. Kildare,2021-01-01,2021,2021Q1,2000.0,Existing,Private Letting,40.82,1,59.83,40.82,40.82,8.87,3.70,42.33,0.0,Pitch Roof-Insul.on Rafter,0.55,0.36,0.49,2.80,3.00,0.15,Medium,Masonry,No,100.0,2.0,0.0,0.0,0.0,2.0,0.0,1.0,Natural vent.,0.0,85.60,0.0,1,1,0,0,None,85.60,110.0,Factory Insulated,0,0,0.00,Unknown,Detached house,Heating Oil,Heating Oil,290.10,D2,old_15_band,0
2,3,Kildare,9,Co. Kildare,2021-01-01,2021,2021Q1,2000.0,Existing,Private Letting,40.82,1,59.83,40.82,40.82,8.87,3.70,43.04,0.0,Pitch Roof-Insul.on Rafter,0.55,0.36,0.49,2.80,3.00,0.15,Medium,Masonry,No,100.0,2.0,0.0,0.0,0.0,2.0,0.0,1.0,Natural vent.,0.0,85.60,0.0,1,1,0,0,None,85.60,110.0,Factory Insulated,0,0,0.00,Unknown,Detached house,Heating Oil,Heating Oil,290.33,D2,old_15_band,0
3,4,Cork,4,Co. Cork,2021-01-01,2021,2021Q1,1930.0,Existing,Major Renovation,147.70,2,114.06,99.66,105.30,30.56,1.85,18.35,0.0,Pitch Roof-Insul.on Ceiling,0.24,0.18,0.11,1.25,1.40,0.15,Medium,Masonry,No,100.0,2.0,0.0,0.0,1.0,0.0,0.0,0.0,Bal.whole mech.vent heat recvr,90.0,464.67,0.1,3,3,0,0,None,257.63,230.0,None,0,0,2169.06,Other,Semi-detached house,Electricity,Electricity,24.52,A1,old_15_band,1
4,5,Laois,11,Co. Laois,2021-01-02,2021,2021Q1,1850.0,Existing,Sale,71.89,1,99.73,71.89,71.89,6.40,1.75,25.59,0.0,Pitch Roof-Insul.on Ceiling,2.10,2.30,0.94,4.48,1.71,0.15,High,Masonry,No,13.0,1.0,0.0,2.0,1.0,0.0,0.0,1.0,Natural vent.,0.0,45.00,0.1,1,2,0,0,None,45.00,95.0,None,0,0,0.00,Stone,Detached house,Solid Multi-Fuel,Solid Multi-Fuel,1200.95,G,old_15_band,0


In [19]:
# Saving the clean BER table.

save_folder = clean + r'\ber'
os.makedirs(save_folder, exist_ok=True)

ber.to_csv(save_folder + r'\ber_clean.csv', index=False)
print('Saved', ber.shape, 'to', save_folder + r'\ber_clean.csv')

Saved (569058, 60) to C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project\dataSet\clean_dataset\ber\ber_clean.csv
